# STORM-PhysNet — Master Reproduction Notebook

This notebook accompanies the **conference** and **IEEE Access** STORM-PhysNet papers.

**What it does**
- Loads the chronological GOES–OMNI split used in the manuscripts
- Trains main models / ablations in DEMO_MODE (short) or full protocol
- Evaluates Prediction Efficiency (PE_clim)
- Loads released `results/*.csv` tables that match the paper numbers
- Provides scaffolds for noise robustness and GRASP transfer

**Honesty / scope**
- `DEMO_MODE = True` (default): short runs for pipeline checks only.
- `DEMO_MODE = False`: full 15-seed protocol — heavy GPU time; the **published PE tables** come from the multi-account training campaign, not from a single notebook run.
- Seed-level CSVs for Access-only controls (wider delay, bagged Transformer) are under `results/`.
- Full training logs / optional checkpoints live on the project Google Drive (see Data Availability).
- `src/data/synthetic_generator.py` and `storm_augmentor.py` are **not** used for reported GOES/OMNI/GRASP tables (`use_real_data: true`).
- Transformer baseline uses default hyperparameters (`d_model=64`, 3 layers, 4 heads) and is **not** capacity-matched to STORM (`d_model=128`, 2 layers).


## 1. Environment Setup


In [ ]:
!nvidia-smi

import os
if not os.path.isdir("src"):
    !git clone https://github.com/bnsama29-cloud/STORM-PhysNet.git
    %cd STORM-PhysNet
else:
    print("Already inside the repo.")

!pip install -q cdflib "numpy<2" pandas scikit-learn pyyaml tqdm matplotlib seaborn torch


## 2. Imports and Global Configuration


In [ ]:
import os
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.cdf_reader import read_goes_directory, read_wind_directory
from src.data.preprocessor import Preprocessor
from src.data.dataloader import make_dataloaders
from src.training.trainer import Trainer
from src.evaluation.metrics import prediction_efficiency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BASE_SEED = 42
SEEDS = list(range(42, 57))  # fifteen seeds used in the papers
torch.manual_seed(BASE_SEED)
np.random.seed(BASE_SEED)

DEMO_MODE = True          # set False only for full paper-scale training
DEMO_EPOCHS = 5
FULL_EPOCHS = 40

print(f"DEMO_MODE = {DEMO_MODE}")
print("Transformer baseline is NOT capacity-matched to STORM (see paper Methods).")


## 3. Load Configuration


In [ ]:
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("forecast_horizons:", config["data"]["forecast_horizons"])
print("sequence_length  :", config["data"]["sequence_length"])
print("STORM d_model    :", config["model"]["d_model"])
print("use_real_data    :", config["data"].get("use_real_data", True))

config.setdefault("training", {})
ckpt = str(config["training"].get("checkpoint_dir", "checkpoints")).replace("\\", "/")
if ckpt.startswith("/kaggle") or "\" in str(config["training"].get("checkpoint_dir", "")):
    ckpt = "checkpoints"
config["training"]["checkpoint_dir"] = ckpt
config["training"]["log_dir"] = "logs"
Path("checkpoints").mkdir(exist_ok=True)
Path("logs").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)


## 4. Data Loading (Chronological Split)


In [ ]:
print("Loading GOES and OMNI from datasets/ (if present)...")

DATA_READY = False
n_sw_features = None
train_loader = val_loader = test_loader = None

try:
    goes_df = read_goes_directory(config["data"].get("goes_cdf_dir", "datasets/goes"))
    wind_df = read_wind_directory(config["data"].get("wind_cdf_dir", "datasets/omni"))
    raw_df = goes_df.join(wind_df, how="inner")
    print(f"Joined dataframe shape: {raw_df.shape}")

    preprocessor = Preprocessor(year_split=config["data"].get("year_split", None))
    train_df, val_df, test_df = preprocessor.fit_transform(raw_df)

    train_loader, val_loader, test_loader = make_dataloaders(
        train_df, val_df, test_df,
        seq_len=config["data"]["sequence_length"],
        batch_size=config["training"].get("batch_size", config["data"].get("batch_size", 64)),
    )
    try:
        batch0 = next(iter(train_loader))
        n_sw_features = int(batch0["x_sw"].shape[-1])
    except Exception:
        n_sw_features = 16
    print(f"Dataloaders OK | n_sw_features={n_sw_features}")
    DATA_READY = True
except Exception as e:
    print("Data loading failed:", e)
    print("Training cells will be skipped; you can still load results/*.csv")
    DATA_READY = False


## 5. Training Helper (uses Trainer.fit)


In [ ]:
def run_training(name, model_type="storm_physnet", gate_type="bz",
                 no_delay=False, no_physics=False, seed=BASE_SEED):
    print("=" * 70)
    print(f"Training: {name} | seed={seed} | type={model_type}")
    print("=" * 70)

    if not DATA_READY:
        print("Data not available - skipping training.")
        return None

    cfg = yaml.safe_load(yaml.dump(config))
    cfg["model_type"] = model_type
    cfg.setdefault("model", {})
    cfg["model"]["gate_type"] = gate_type

    if no_delay:
        cfg["ablation"] = "no_delay"
    elif no_physics:
        cfg["ablation"] = "no_physics"
    elif gate_type in (None, "none"):
        cfg["ablation"] = "no_bz_gate"
        cfg["model"]["gate_type"] = "bz"
    else:
        cfg["ablation"] = "none"

    cfg["training"]["seed"] = int(seed)
    cfg["training"]["epochs"] = DEMO_EPOCHS if DEMO_MODE else FULL_EPOCHS
    cfg["training"]["checkpoint_dir"] = f"checkpoints/{name}/seed_{seed}"
    cfg["training"]["log_dir"] = f"logs/{name}"
    Path(cfg["training"]["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)

    torch.manual_seed(seed)
    np.random.seed(seed)

    trainer = Trainer(cfg)
    try:
        model = trainer.fit(
            train_loader,
            val_loader,
            n_sw_features=n_sw_features,
            use_ensemble=False,
        )
        print(f"Finished training {name}")
        return model
    except Exception as e:
        print(f"Training failed for {name}: {e}")
        return None


## 6. Train Main Models (optional DEMO)


In [ ]:
# Optional single-seed demo trains. For paper tables, prefer results/*.csv.
run_training("transformer", model_type="transformer")
run_training("lstm", model_type="lstm")
run_training("storm_bz", model_type="storm_physnet", gate_type="bz")


## 7. Train Ablations (optional DEMO)


In [ ]:
run_training("storm_no_delay", model_type="storm_physnet", gate_type="bz", no_delay=True)
run_training("storm_no_gate", model_type="storm_physnet", gate_type="none")
run_training("storm_no_physics", model_type="storm_physnet", gate_type="bz", no_physics=True)


## 8. Wider-Delay Experiment (Access)


In [ ]:
def run_wider_delay_experiment(seed=42, upper_bound=2.0):
    print(f"WIDER DELAY | seed={seed} | upper_bound={upper_bound}h")
    if not DATA_READY:
        print("Data not available - skip.")
        return
    cfg = yaml.safe_load(yaml.dump(config))
    cfg["model_type"] = "storm_physnet"
    cfg.setdefault("model", {})
    cfg["model"]["delay"] = {"enabled": True}
    cfg["model"]["delay_min"] = 0.5
    cfg["model"]["delay_max"] = float(upper_bound)
    cfg["training"]["seed"] = int(seed)
    cfg["training"]["epochs"] = DEMO_EPOCHS if DEMO_MODE else FULL_EPOCHS
    cfg["training"]["checkpoint_dir"] = f"checkpoints/wider_delay_{upper_bound}h/seed_{seed}"
    Path(cfg["training"]["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    trainer = Trainer(cfg)
    try:
        trainer.fit(train_loader, val_loader, n_sw_features=n_sw_features, use_ensemble=False)
        print("Wider-delay training finished.")
    except Exception as e:
        print("Wider-delay failed:", e)

run_wider_delay_experiment(seed=BASE_SEED, upper_bound=2.0)
print("Full multi-bound results are in results/wider_delay_results.csv")


## 9. Bagged Transformer Control (Access)


In [ ]:
def run_bagged_transformer_experiment(seed=42):
    print(f"BAGGED TF CONTROL | seed={seed}")
    if not DATA_READY:
        print("Data not available - skip.")
        return
    cfg = yaml.safe_load(yaml.dump(config))
    cfg["model_type"] = "transformer"
    cfg["training"]["seed"] = int(seed)
    cfg["training"]["epochs"] = DEMO_EPOCHS if DEMO_MODE else FULL_EPOCHS
    cfg["training"]["checkpoint_dir"] = f"checkpoints/bagged_tf/seed_{seed}"
    Path(cfg["training"]["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    trainer = Trainer(cfg)
    try:
        trainer.fit(train_loader, val_loader, n_sw_features=n_sw_features, use_ensemble=False)
        print("Bagged-TF seed training finished.")
    except Exception as e:
        print("Bagged-TF failed:", e)

run_bagged_transformer_experiment(seed=BASE_SEED)
print("Full bagged-TF seed table: results/bagged_tf_results.csv")


## 10. Evaluation helper (no undefined symbols)


In [ ]:
def evaluate_checkpoint(ckpt_path, model_type="storm_physnet", gate_type="bz", ablation="none"):
    if not DATA_READY:
        print("No test data.")
        return None
    if not os.path.exists(ckpt_path):
        print(f"Checkpoint not found: {ckpt_path}")
        return None

    cfg = yaml.safe_load(yaml.dump(config))
    cfg["model_type"] = model_type
    cfg.setdefault("model", {})
    cfg["model"]["gate_type"] = gate_type
    cfg["ablation"] = ablation
    trainer = Trainer(cfg)
    model = trainer.build_model(n_sw_features).to(device)

    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
    elif isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict(state["state_dict"])
    else:
        model.load_state_dict(state)
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in test_loader:
            x_sw = batch["x_sw"].to(device)
            x_flux = batch["x_flux"].to(device)
            y = batch["y_flux"]
            yp = batch.get("y_persist")
            if yp is not None:
                yp = yp.to(device)
            try:
                out = model(x_sw, x_flux, y_persist=yp) if yp is not None else model(x_sw, x_flux)
            except TypeError:
                out = model(x_sw, x_flux)
            pred = out["flux_pred"] if isinstance(out, dict) else out
            y_true.append(y.numpy())
            y_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)

    metrics = {}
    labels = ["PE_45min", "PE_6h", "PE_12h"]
    for i, lab in enumerate(labels):
        if y_true.shape[1] > i:
            metrics[lab] = float(prediction_efficiency(y_true[:, i], y_pred[:, i]))
    print(metrics)
    return metrics


## 11. Load paper result tables (recommended)


In [ ]:
paths = {
    "ablation": Path("results/ablation_final_table.csv"),
    "all": Path("results/all_results.csv"),
    "wider": Path("results/wider_delay_results.csv"),
    "bagged_tf": Path("results/bagged_tf_results.csv"),
    "summary": Path("results/summary.json"),
}

for k, p in paths.items():
    if p.exists():
        print(f"\n=== {p} ===")
        if p.suffix == ".json":
            print(p.read_text()[:2000])
        else:
            df = pd.read_csv(p)
            print(df.head(12).to_string(index=False))
            print("rows:", len(df))
    else:
        print(f"Missing: {p}")


## 12. Multi-seed scaffold (only if DEMO_MODE=False)


In [ ]:
def run_multi_seed(name, model_type="storm_physnet", gate_type="bz",
                   no_delay=False, no_physics=False):
    if DEMO_MODE:
        print("DEMO_MODE=True -> refusing full multi-seed loop. Set DEMO_MODE=False to run.")
        return None
    rows = []
    for seed in SEEDS:
        run_training(name, model_type=model_type, gate_type=gate_type,
                     no_delay=no_delay, no_physics=no_physics, seed=seed)
        rows.append({"seed": seed, "name": name})
    df = pd.DataFrame(rows)
    out = Path("results") / f"{name}_multiseed_runs.csv"
    df.to_csv(out, index=False)
    print("Wrote", out)
    return df

print("Multi-seed helper defined. Published PE means come from results/*.csv / Drive campaign.")


## 13. Noise / GRASP scaffolds


In [ ]:
def noise_robustness_experiment():
    print("Noise robustness scaffold (Access paper).")
    print("Add N(0, sigma^2) to standardized SW inputs at test time; score PE_45min and PE_6h.")

def grasp_transfer_experiment():
    print("GRASP transfer scaffold (both papers).")
    print("Freeze encoder, fine-tune residual heads on GRASP train split; evaluate PE.")

noise_robustness_experiment()
grasp_transfer_experiment()


## 14. Summary for reviewers

| Item | Status |
|------|--------|
| Chronological split | Implemented when datasets/ present |
| Transformer / LSTM / STORM-Bz | Train helper via `Trainer.fit` |
| Ablations | Train helper |
| Wider delay / bagged TF | Helpers + `results/*.csv` |
| Paper PE tables | **Load `results/` CSVs** (authoritative) |
| Full 15-seed retrain | Optional; not required to read paper numbers |
| Capacity mismatch | Documented (TF defaults vs STORM config) |
| Synthetic generators | Not used for reported tables |

**Drive (full logs / checkpoints):** see Data Availability in the papers.
